In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")


# CONFIGURAÇÕES GERAIS


N_CLUSTERS_GEOGRAFICOS = 5
RANDOM_STATE = 42
TEST_SIZE = 0.2 
MAX_SAMPLE_CONFIGS = 500_000
#limitar o número de linhas usadas durante as configs de testes q fui avaliando, problema de performance.
MAX_SAMPLE_FINAL = 1_000_000
#Limita o tamanho da amostra usada na fase final, qnd já foi escolhida a melhor config.


# CARREGAMENTO E PRÉ-PROCESSAMENTO INICIAL

def carregar_dados(caminho_ficheiro: str) -> pd.DataFrame:
    df = pd.read_csv(caminho_ficheiro)
    registos_antes = df.shape[0]

    df = df[df['vmc_kmh'] > 1].copy()
    registos_depois = df.shape[0]

    print(f"Removidos {registos_antes - registos_depois:,} registos com vmc_kmh <= 1 km/h.")
    print(f"Total de {registos_depois:,} registos após filtragem.")
    return df


def criar_atributos_base(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()

    df2[['latitude', 'longitude']] = df2['zona_id'].str.split('_', expand=True).astype(float)

    df2['periodo'] = np.where(
        df2['classe_hora'].str.lower().eq('pico'),
        'pico',
        'nao_pico'
    )

    df2['eh_feriado'] = df2['feriado'].astype(int)

    df2['nivel_atraso'] = pd.cut(
        df2['atraso_medio_waze'],
        bins=[-np.inf, 50, 100, 150, 250, np.inf],
        labels=['baixo', 'medio', 'alto', 'muito_alto', 'critico']
    )

    return df2


def aplicar_clusterizacao_geografica(df: pd.DataFrame, n_clusters: int) -> pd.DataFrame:
    coords = df[['latitude', 'longitude']].dropna()
    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)

    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
    clusters = kmeans.fit_predict(coords_scaled)

    df2 = df.copy()
    df2.loc[coords.index, 'zona_cluster'] = clusters.astype(str)

    print(f"Clusterização geográfica aplicada ({n_clusters} clusters).")
    return df2



# CONFIGS DOS ATRIBUTOS


def definir_configuracoes_atributos():
    return {
        'config_base': {
            'categoricos': ['tipo_dia', 'periodo', 'route_number'],
            'numericos': ['dia_semana', 'eh_feriado', 'atraso_medio_waze']
        },
        'config_geografica': {
            'categoricos': ['tipo_dia', 'periodo', 'route_number'],
            'numericos': ['dia_semana', 'eh_feriado', 'atraso_medio_waze', 'latitude', 'longitude']
        },
        'config_cluster': {
            'categoricos': ['tipo_dia', 'periodo', 'route_number', 'zona_cluster'],
            'numericos': ['dia_semana', 'eh_feriado', 'atraso_medio_waze']
        },
        'config_chuva': {
            'categoricos': ['tipo_dia', 'periodo', 'route_number', 'classe_chuva'],
            'numericos': ['dia_semana', 'eh_feriado', 'atraso_medio_waze', 'latitude', 'longitude']
        },
        'config_atraso_categorizado': {
            'categoricos': ['tipo_dia', 'periodo', 'route_number', 'zona_cluster', 'nivel_atraso'],
            'numericos': ['dia_semana', 'eh_feriado']
        },
        'config_completa': {
            'categoricos': ['tipo_dia', 'periodo', 'route_number', 'classe_chuva', 'zona_cluster'],
            'numericos': ['dia_semana', 'eh_feriado', 'atraso_medio_waze']
        }
    }



# PREPARAÇÃO DOS DADOS


def preparar_dados(df, config, max_sample):
    cols = config['categoricos'] + config['numericos']
    df2 = df.dropna(subset=cols + ['vmc_kmh'])

    if len(df2) > max_sample:
        df2 = df2.sample(n=max_sample, random_state=RANDOM_STATE)

    X = df2[cols].copy()
    y = df2['vmc_kmh'].copy()

    for c in config['categoricos']:
        X[c] = X[c].astype('category').cat.codes

    return X, y, cols


# MÉTRICAS P AVALIAÇÃO


def avaliar(y_true, y_pred):
    mask = y_true > 0
    y_t = y_true[mask]
    y_p = y_pred[mask]

    mape = np.mean(np.abs((y_t - y_p) / y_t)) * 100 if len(y_t) else np.inf

    return {
        'r2': r2_score(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'mape': mape
    }



# TREINO E AVALIAÇAO DOS MODELOS


def treinar_modelos(X_train, y_train, X_test, y_test):
    modelos = {
        'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15,
                                               min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                                       max_depth=5, random_state=RANDOM_STATE),
        'Regressão Linear': LinearRegression()
    }

    resultados = []
    for nome, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        preds = modelo.predict(X_test)
        m = avaliar(y_test, preds)
        m['modelo'] = nome
        resultados.append(m)

    df_res = pd.DataFrame(resultados).sort_values('r2', ascending=False)
    melhor_nome = df_res.iloc[0]['modelo']
    return modelos[melhor_nome], melhor_nome, df_res



# IMPORTÂNCIA DOS ATRIBUTOS


def importancia_atributos(modelo, features, titulo):
    if not hasattr(modelo, 'feature_importances_'):
        print("Modelo não suporta feature_importances_.")
        return None

    imp = pd.DataFrame({
        'Atributo': features,
        'Importancia': modelo.feature_importances_
    }).sort_values('Importancia', ascending=False)

    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importancia', y='Atributo', data=imp.head(15))
    plt.title(titulo)
    plt.tight_layout()
    plt.show()

    return imp


# EXECUÇÃO DA MODELAÇÃO COMPLETA


def executar():
    df = carregar_dados('df_modelo_export.csv')
    df = criar_atributos_base(df)
    df = aplicar_clusterizacao_geografica(df, N_CLUSTERS_GEOGRAFICOS)

    configs = definir_configuracoes_atributos()
    resultados = {}

    for nome, cfg in configs.items():
        print(f"\nA avaliar {nome}")
        X, y, _ = preparar_dados(df, cfg, MAX_SAMPLE_CONFIGS)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

        _, melhor_modelo_nome, df_result = treinar_modelos(X_train, y_train, X_test, y_test)
        resultados[nome] = df_result
        print(df_result)

    # Escolher a melhor configuração
    melhor_config = max(resultados, key=lambda k: resultados[k].iloc[0]['r2'])
    print(f"\nMelhor configuração: {melhor_config}")

    # Treino final
    X_final, y_final, feat_final = preparar_dados(df, configs[melhor_config], MAX_SAMPLE_FINAL)
    X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    param_grid = {
        'n_estimators': [200, 300],
        'max_depth': [20, 25],
        'min_samples_leaf': [2, 4]
    }

    grid = GridSearchCV(
        RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        param_grid, scoring='r2', cv=3, verbose=1
    )
    grid.fit(X_train, y_train)

    modelo_final = grid.best_estimator_
    preds_final = modelo_final.predict(X_test)
    metricas = avaliar(y_test, preds_final)

    print("\nRESULTADOS FINAIS")
    print(metricas)

    importancia_atributos(modelo_final, feat_final, "Importância dos Atributos - Modelo Final")

    return modelo_final, metricas


if __name__ == "__main__":
    executar()
